In [1]:
"""
QUICKBITE PRODUCT ANALYTICS PLATFORM
Phase 2 - Notebook 10: Anomaly Detection
==================================================================
Purpose: Detect anomalies in key business metrics to identify
issues, opportunities, and operational problems in real-time.

Key Questions:
1. Which days/weeks show unusual patterns in orders, GMV, or cancellations?
2. Is there a structural break in our metrics?
3. What caused the anomalies?
4. How can we build a real-time monitoring system?

Author: Senior Product Analytics Team
Date: 2026-07-28
"""

'\nQUICKBITE PRODUCT ANALYTICS PLATFORM\nPhase 2 - Notebook 10: Anomaly Detection\n==================================================================\nPurpose: Detect anomalies in key business metrics to identify\nissues, opportunities, and operational problems in real-time.\n\nKey Questions:\n1. Which days/weeks show unusual patterns in orders, GMV, or cancellations?\n2. Is there a structural break in our metrics?\n3. What caused the anomalies?\n4. How can we build a real-time monitoring system?\n\nAuthor: Senior Product Analytics Team\nDate: 2026-07-28\n'

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
from scipy import stats
from scipy.signal import find_peaks
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Set visualization style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

In [ ]:
print("="*80)
print("QUICKBITE ANOMALY DETECTION")
print("="*80)

---------------------------------------------------------------------
1. LOAD CLEANED DATA
---------------------------------------------------------------------

In [ ]:
print("\n📂 Loading cleaned data...")

In [ ]:
orders = pd.read_csv('../outputs/cleaned_data/orders_cleaned.csv')
users = pd.read_csv('../outputs/cleaned_data/users_cleaned.csv')
payments = pd.read_csv('../outputs/cleaned_data/payments_cleaned.csv')
cities = pd.read_csv('../data/cities.csv')
weather = pd.read_csv('../data/weather.csv')
traffic = pd.read_csv('../data/traffic.csv')

In [ ]:
# Convert dates
orders['order_placed_at'] = pd.to_datetime(orders['order_placed_at'])
users['signup_date'] = pd.to_datetime(users['signup_date'])
payments['processed_at'] = pd.to_datetime(payments['processed_at'])
weather['date'] = pd.to_datetime(weather['date'])
traffic['date'] = pd.to_datetime(traffic['date'])

In [ ]:
print(f"✅ Loaded {len(orders):,} orders")
print(f"✅ Loaded {len(users):,} users")
print(f"✅ Loaded {len(payments):,} payments")

---------------------------------------------------------------------
2. PREPARE DAILY METRICS
---------------------------------------------------------------------

In [ ]:
print("\n" + "="*80)
print("PREPARING DAILY METRICS")
print("="*80)

In [ ]:
# Create daily aggregates
daily_metrics = orders.groupby(orders['order_placed_at'].dt.date).agg({
    'order_id': 'count',
    'total_amount': 'sum',
    'order_status': lambda x: (x == 'cancelled').sum() / len(x) * 100
}).reset_index()

In [ ]:
daily_metrics.columns = ['date', 'orders', 'gmv', 'cancellation_rate']

In [ ]:
# Add delivered orders count
delivered_orders = orders[orders['order_status'] == 'delivered']
daily_delivered = delivered_orders.groupby(delivered_orders['order_placed_at'].dt.date).size().reset_index()
daily_delivered.columns = ['date', 'delivered_orders']
daily_metrics = daily_metrics.merge(daily_delivered, on='date', how='left')

In [ ]:
# Add AOV
daily_metrics['aov'] = daily_metrics['gmv'] / daily_metrics['delivered_orders']

In [ ]:
# Add user activity
daily_users = orders.groupby(orders['order_placed_at'].dt.date)['user_id'].nunique().reset_index()
daily_users.columns = ['date', 'active_users']
daily_metrics = daily_metrics.merge(daily_users, on='date', how='left')

In [ ]:
# Add payment metrics
payments['date'] = payments['processed_at'].dt.date
daily_payments = payments.groupby('date').agg({
    'payment_id': 'count',
    'payment_status': lambda x: (x == 'failed').sum() / len(x) * 100
}).reset_index()
daily_payments.columns = ['date', 'payment_attempts', 'payment_failure_rate']
daily_metrics = daily_metrics.merge(daily_payments, on='date', how='left')

In [ ]:
# Fill missing values
daily_metrics = daily_metrics.fillna(0)

In [ ]:
# Add weather and traffic data
weather_daily = weather.groupby('date')['condition'].agg(lambda x: x.mode()[0] if len(x) > 0 else 'clear').reset_index()
traffic_daily = traffic.groupby('date')['traffic_index'].mean().reset_index()

In [ ]:
daily_metrics = daily_metrics.merge(weather_daily, on='date', how='left')
daily_metrics = daily_metrics.merge(traffic_daily, on='date', how='left')

In [ ]:
# Add date features
daily_metrics['date'] = pd.to_datetime(daily_metrics['date'])
daily_metrics['day_of_week'] = daily_metrics['date'].dt.dayofweek
daily_metrics['month'] = daily_metrics['date'].dt.month
daily_metrics['is_weekend'] = daily_metrics['day_of_week'] >= 5

In [ ]:
print(f"✅ Created {len(daily_metrics)} days of metrics")

---------------------------------------------------------------------
3. STATISTICAL ANOMALY DETECTION
---------------------------------------------------------------------

In [ ]:
print("\n" + "="*80)
print("STATISTICAL ANOMALY DETECTION")
print("="*80)

In [ ]:
def detect_statistical_anomalies(series, threshold=3, window=30):
    """
    Detect anomalies using Z-score method with rolling statistics
    """
    rolling_mean = series.rolling(window=window, center=True).mean()
    rolling_std = series.rolling(window=window, center=True).std()
    
    z_scores = (series - rolling_mean) / rolling_std
    anomalies = abs(z_scores) > threshold
    
    return anomalies, z_scores

In [ ]:
# Detect anomalies in key metrics
metrics_to_check = ['orders', 'gmv', 'cancellation_rate', 'aov', 'active_users', 'payment_failure_rate']
anomaly_results = {}

In [ ]:
for metric in metrics_to_check:
    if metric in daily_metrics.columns:
        anomalies, z_scores = detect_statistical_anomalies(
            daily_metrics[metric], 
            threshold=3, 
            window=14
        )
        anomaly_results[metric] = {
            'anomalies': anomalies,
            'z_scores': z_scores,
            'count': anomalies.sum()
        }
        print(f"\n📊 {metric.upper()}:")
        print(f"  • Anomalies detected: {anomalies.sum()}")
        print(f"  • Anomaly rate: {anomalies.sum() / len(anomalies) * 100:.2f}%")

In [ ]:
# Visualize anomalies for each metric
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('Statistical Anomaly Detection - Key Metrics', fontsize=16, fontweight='bold')

In [ ]:
for idx, (metric, results) in enumerate(anomaly_results.items()):
    if idx >= 6:
        break
    
    ax = axes[idx // 3, idx % 3]
    
    # Plot the metric
    ax.plot(daily_metrics['date'], daily_metrics[metric], color='#3498db', linewidth=1.5, label=metric)
    
    # Highlight anomalies
    anomaly_dates = daily_metrics['date'][results['anomalies']]
    anomaly_values = daily_metrics[metric][results['anomalies']]
    
    if len(anomaly_dates) > 0:
        ax.scatter(anomaly_dates, anomaly_values, color='red', s=50, label='Anomalies', zorder=5)
    
    ax.set_title(f'{metric.upper()} - {results["count"]} anomalies')
    ax.set_xlabel('Date')
    ax.set_ylabel(metric)
    ax.legend()
    ax.tick_params(axis='x', rotation=45)
    ax.grid(True, alpha=0.3)

In [ ]:
plt.tight_layout()
plt.savefig('../outputs/visualizations/statistical_anomalies.png', dpi=300, bbox_inches='tight')
plt.show()

---------------------------------------------------------------------
4. ISOLATION FOREST ANOMALY DETECTION
---------------------------------------------------------------------

In [ ]:
print("\n" + "="*80)
print("ISOLATION FOREST ANOMALY DETECTION")
print("="*80)

In [ ]:
# Prepare features for Isolation Forest
isolation_features = ['orders', 'gmv', 'cancellation_rate', 'aov', 'active_users', 'payment_failure_rate']
X_iforest = daily_metrics[isolation_features].copy()

In [ ]:
# Scale features
scaler_iforest = StandardScaler()
X_iforest_scaled = scaler_iforest.fit_transform(X_iforest.fillna(0))

In [ ]:
# Train Isolation Forest
iso_forest = IsolationForest(
    contamination=0.05,  # Expected 5% anomalies
    random_state=42,
    n_estimators=100
)
iso_forest.fit(X_iforest_scaled)

In [ ]:
# Predict anomalies
isolation_anomalies = iso_forest.predict(X_iforest_scaled) == -1
daily_metrics['isolation_anomaly'] = isolation_anomalies

In [ ]:
print(f"\n📊 Isolation Forest Results:")
print(f"  • Anomalies detected: {isolation_anomalies.sum()}")
print(f"  • Anomaly rate: {isolation_anomalies.sum() / len(isolation_anomalies) * 100:.2f}%")

In [ ]:
# Combine with statistical anomalies
daily_metrics['combined_anomaly'] = (
    daily_metrics['orders_anomaly'] | 
    daily_metrics['gmv_anomaly'] | 
    daily_metrics['cancellation_rate_anomaly'] |
    daily_metrics['isolation_anomaly']
)

In [ ]:
print(f"\n📊 Combined Anomalies:")
print(f"  • Total unique anomalous days: {daily_metrics['combined_anomaly'].sum()}")

In [ ]:
# Visualize Isolation Forest results
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Isolation Forest Anomaly Detection', fontsize=14, fontweight='bold')

In [ ]:
# 4.1 Anomaly distribution
ax = axes[0, 0]
anomaly_counts = pd.DataFrame({
    'Type': ['Statistical Only', 'Isolation Forest Only', 'Both'],
    'Count': [
        (daily_metrics['orders_anomaly'] | daily_metrics['gmv_anomaly']) & ~daily_metrics['isolation_anomaly'],
        daily_metrics['isolation_anomaly'] & ~(daily_metrics['orders_anomaly'] | daily_metrics['gmv_anomaly']),
        daily_metrics['isolation_anomaly'] & (daily_metrics['orders_anomaly'] | daily_metrics['gmv_anomaly'])
    ]
})
anomaly_counts['Count'] = anomaly_counts['Count'].apply(sum)
anomaly_counts.plot(kind='bar', ax=ax, color=['#3498db', '#e67e22', '#e74c3c'], alpha=0.7)
ax.set_title('Anomaly Detection Method Comparison')
ax.set_xlabel('Method')
ax.set_ylabel('Number of Anomalies')
ax.tick_params(axis='x', rotation=0)

In [ ]:
# 4.2 Anomaly heatmap over time
ax = axes[0, 1]
anomaly_matrix = daily_metrics[['orders_anomaly', 'gmv_anomaly', 'cancellation_rate_anomaly', 
                                 'aov_anomaly', 'active_users_anomaly', 'isolation_anomaly']].T
sns.heatmap(anomaly_matrix, ax=ax, cmap='RdYlGn_r', cbar_kws={'label': 'Anomaly Detected'})
ax.set_title('Anomaly Heatmap Over Time')
ax.set_xlabel('Date Index')
ax.set_ylabel('Metric')

In [ ]:
# 4.3 Feature contribution to isolation forest anomalies
ax = axes[1, 0]
feature_importance_iforest = pd.DataFrame({
    'feature': isolation_features,
    'importance': np.abs(iso_forest.feature_importances_)
}).sort_values('importance', ascending=False)

In [ ]:
ax.barh(feature_importance_iforest['feature'], feature_importance_iforest['importance'], 
        color='#9b59b6', alpha=0.7)
ax.set_title('Isolation Forest Feature Importance')
ax.set_xlabel('Importance')
ax.invert_yaxis()

In [ ]:
# 4.4 Anomaly severity scores
ax = axes[1, 1]
anomaly_scores = -iso_forest.score_samples(X_iforest_scaled)  # Lower = more anomalous
daily_metrics['anomaly_score'] = anomaly_scores

In [ ]:
ax.scatter(daily_metrics['date'], daily_metrics['anomaly_score'], 
           c=daily_metrics['isolation_anomaly'], cmap='RdYlGn_r', alpha=0.6)
ax.axhline(y=np.percentile(anomaly_scores, 95), color='red', linestyle='--', 
           label='95th Percentile')
ax.set_title('Anomaly Scores Over Time')
ax.set_xlabel('Date')
ax.set_ylabel('Anomaly Score')
ax.legend()
ax.tick_params(axis='x', rotation=45)

In [ ]:
plt.tight_layout()
plt.savefig('../outputs/visualizations/isolation_forest_anomalies.png', dpi=300, bbox_inches='tight')
plt.show()

---------------------------------------------------------------------
5. ANOMALY ROOT CAUSE ANALYSIS
---------------------------------------------------------------------

In [ ]:
print("\n" + "="*80)
print("ANOMALY ROOT CAUSE ANALYSIS")
print("="*80)

In [ ]:
# Identify anomalous days
anomaly_days = daily_metrics[daily_metrics['combined_anomaly'] == True]

In [ ]:
if len(anomaly_days) > 0:
    print(f"\n📊 Anomaly Days Analysis:")
    print(f"  • Total anomalous days: {len(anomaly_days)}")
    print(f"  • Date range: {anomaly_days['date'].min()} to {anomaly_days['date'].max()}")
    
    # Analyze patterns in anomaly days
    print("\n📊 Anomaly Day Patterns:")
    print(f"  • Weekend days: {anomaly_days['is_weekend'].sum()} ({anomaly_days['is_weekend'].sum()/len(anomaly_days)*100:.1f}%)")
    print(f"  • Weekday vs Weekend avg orders: {anomaly_days[~anomaly_days['is_weekend']]['orders'].mean():.0f} vs {anomaly_days[anomaly_days['is_weekend']]['orders'].mean():.0f}")
    
    # Weather correlation
    weather_counts = anomaly_days['condition'].value_counts()
    print(f"\n📊 Weather Conditions on Anomaly Days:")
    for condition, count in weather_counts.items():
        print(f"  • {condition}: {count} days ({count/len(anomaly_days)*100:.1f}%)")
    
    # Traffic correlation
    print(f"\n📊 Traffic Index on Anomaly Days:")
    print(f"  • Avg traffic: {anomaly_days['traffic_index'].mean():.2f}")
    print(f"  • Normal avg traffic: {daily_metrics[~daily_metrics['combined_anomaly']]['traffic_index'].mean():.2f}")
    
    # Anomaly type classification
    anomaly_days['anomaly_type'] = 'Unknown'
    
    # Classify anomalies
    anomaly_days.loc[anomaly_days['orders'] > daily_metrics['orders'].mean() + 2*daily_metrics['orders'].std(), 'anomaly_type'] = 'Order Spike'
    anomaly_days.loc[anomaly_days['orders'] < daily_metrics['orders'].mean() - 2*daily_metrics['orders'].std(), 'anomaly_type'] = 'Order Drop'
    anomaly_days.loc[anomaly_days['gmv'] > daily_metrics['gmv'].mean() + 2*daily_metrics['gmv'].std(), 'anomaly_type'] = 'Revenue Spike'
    anomaly_days.loc[anomaly_days['cancellation_rate'] > daily_metrics['cancellation_rate'].mean() + 2*daily_metrics['cancellation_rate'].std(), 'anomaly_type'] = 'Cancellation Spike'
    anomaly_days.loc[anomaly_days['payment_failure_rate'] > daily_metrics['payment_failure_rate'].mean() + 2*daily_metrics['payment_failure_rate'].std(), 'anomaly_type'] = 'Payment Failure Spike'
    
    # Count anomaly types
    print(f"\n📊 Anomaly Type Distribution:")
    anomaly_type_counts = anomaly_days['anomaly_type'].value_counts()
    for atype, count in anomaly_type_counts.items():
        print(f"  • {atype}: {count} days ({count/len(anomaly_days)*100:.1f}%)")

---------------------------------------------------------------------
6. STRUCTURAL BREAK ANALYSIS
---------------------------------------------------------------------

In [ ]:
print("\n" + "="*80)
print("STRUCTURAL BREAK ANALYSIS")
print("="*80)

In [ ]:
def detect_structural_break(series, min_segment_size=10):
    """
    Detect structural breaks using Chow test approximation
    """
    n = len(series)
    results = []
    
    for i in range(min_segment_size, n - min_segment_size):
        # Split into two segments
        segment1 = series[:i]
        segment2 = series[i:]
        
        # Calculate means and variances
        mean1, std1 = segment1.mean(), segment1.std()
        mean2, std2 = segment2.mean(), segment2.std()
        
        # Calculate effect size (Cohen's d)
        pooled_std = np.sqrt(((len(segment1) - 1) * std1**2 + (len(segment2) - 1) * std2**2) / 
                            (len(segment1) + len(segment2) - 2))
        effect_size = (mean2 - mean1) / pooled_std if pooled_std > 0 else 0
        
        # Calculate t-statistic
        se = pooled_std * np.sqrt(1/len(segment1) + 1/len(segment2))
        t_stat = (mean2 - mean1) / se if se > 0 else 0
        
        # P-value approximation
        p_value = 2 * (1 - stats.t.cdf(abs(t_stat), len(segment1) + len(segment2) - 2))
        
        results.append({
            'break_point': i,
            'mean1': mean1,
            'mean2': mean2,
            'difference': mean2 - mean1,
            'effect_size': effect_size,
            't_stat': t_stat,
            'p_value': p_value,
            'significant': p_value < 0.05
        })
    
    return pd.DataFrame(results)

In [ ]:
# Detect breaks in key metrics
break_metrics = ['orders', 'gmv', 'cancellation_rate']

In [ ]:
for metric in break_metrics:
    print(f"\n📊 Structural Break Analysis - {metric.upper()}:")
    
    # Use the last 90 days for faster analysis
    series = daily_metrics[metric].tail(90).reset_index(drop=True)
    
    break_results = detect_structural_break(series, min_segment_size=14)
    
    if len(break_results) > 0:
        significant_breaks = break_results[break_results['significant'] == True]
        
        if len(significant_breaks) > 0:
            largest_break = significant_breaks.loc[significant_breaks['effect_size'].abs().idxmax()]
            
            print(f"  • Largest significant break at index {largest_break['break_point']}")
            print(f"  • Mean before: {largest_break['mean1']:.2f}")
            print(f"  • Mean after: {largest_break['mean2']:.2f}")
            print(f"  • Change: {largest_break['difference']:.2f} ({largest_break['effect_size']:.2f} effect size)")
            print(f"  • P-value: {largest_break['p_value']:.4f}")
            
            # Visualize break
            fig, ax = plt.subplots(figsize=(12, 4))
            
            break_point = largest_break['break_point']
            ax.plot(range(len(series)), series, color='#3498db', linewidth=1.5)
            ax.axvline(x=break_point, color='red', linestyle='--', 
                      label=f'Break point: {break_point}')
            
            # Add segment means
            ax.axhline(y=largest_break['mean1'], xmin=0, xmax=break_point/len(series), 
                      color='green', linestyle='--', label='Mean before')
            ax.axhline(y=largest_break['mean2'], xmin=break_point/len(series), xmax=1, 
                      color='orange', linestyle='--', label='Mean after')
            
            ax.set_title(f'Structural Break in {metric.upper()}')
            ax.set_xlabel('Day (last 90 days)')
            ax.set_ylabel(metric)
            ax.legend()
            ax.grid(True, alpha=0.3)
            
            plt.tight_layout()
            plt.savefig(f'../outputs/visualizations/structural_break_{metric}.png', dpi=300, bbox_inches='tight')
            plt.show()
        else:
            print("  • No significant structural breaks detected")
    else:
        print("  • Insufficient data for break detection")

---------------------------------------------------------------------
7. SEASONAL ANOMALY DETECTION
---------------------------------------------------------------------

In [ ]:
print("\n" + "="*80)
print("SEASONAL ANOMALY DETECTION")
print("="*80)

In [ ]:
def detect_seasonal_anomalies(series, period=7, threshold=3):
    """
    Detect anomalies relative to seasonal patterns
    """
    n = len(series)
    seasonal_means = np.zeros(n)
    
    # Calculate seasonal means
    for i in range(period):
        indices = range(i, n, period)
        if len(indices) > 0:
            seasonal_means[indices] = series.iloc[indices].mean()
    
    # Calculate deviations
    deviations = series - seasonal_means
    std_dev = deviations.std()
    
    # Detect anomalies
    anomalies = abs(deviations) > threshold * std_dev
    
    return anomalies, deviations, seasonal_means

In [ ]:
# Detect seasonal anomalies in orders
seasonal_anomalies, deviations, seasonal_means = detect_seasonal_anomalies(
    daily_metrics['orders'], 
    period=7,  # Weekly seasonality
    threshold=2.5
)

In [ ]:
daily_metrics['seasonal_anomaly'] = seasonal_anomalies

In [ ]:
print(f"\n📊 Seasonal Anomaly Detection Results:")
print(f"  • Seasonal anomalies detected: {seasonal_anomalies.sum()}")
print(f"  • Anomaly rate: {seasonal_anomalies.sum() / len(seasonal_anomalies) * 100:.2f}%")

In [ ]:
# Visualize seasonal anomalies
fig, axes = plt.subplots(2, 1, figsize=(14, 10))
fig.suptitle('Seasonal Anomaly Detection', fontsize=14, fontweight='bold')

In [ ]:
# 7.1 Orders with seasonal pattern
ax = axes[0]
ax.plot(daily_metrics['date'], daily_metrics['orders'], color='#3498db', linewidth=1.5, label='Actual')

In [ ]:
# Highlight seasonal anomalies
anomaly_dates = daily_metrics['date'][seasonal_anomalies]
anomaly_values = daily_metrics['orders'][seasonal_anomalies]
ax.scatter(anomaly_dates, anomaly_values, color='red', s=50, label='Seasonal Anomalies', zorder=5)

In [ ]:
ax.set_title('Orders with Seasonal Anomalies Detected')
ax.set_xlabel('Date')
ax.set_ylabel('Orders')
ax.legend()
ax.tick_params(axis='x', rotation=45)
ax.grid(True, alpha=0.3)

In [ ]:
# 7.2 Deviation from seasonal pattern
ax = axes[1]
ax.fill_between(daily_metrics['date'], -threshold * deviations.std(), threshold * deviations.std(), 
                alpha=0.2, color='#2ecc71', label='Normal Range')
ax.plot(daily_metrics['date'], deviations, color='#e74c3c', linewidth=1.5, label='Deviation')
ax.axhline(y=0, color='black', linestyle='-', alpha=0.5)

In [ ]:
# Highlight anomalies in deviation
deviation_anomalies = abs(deviations) > threshold * deviations.std()
ax.scatter(daily_metrics['date'][deviation_anomalies], deviations[deviation_anomalies], 
           color='red', s=50, label='Anomalies', zorder=5)

In [ ]:
ax.set_title('Deviation from Weekly Seasonal Pattern')
ax.set_xlabel('Date')
ax.set_ylabel('Deviation from Seasonal Mean')
ax.legend()
ax.tick_params(axis='x', rotation=45)
ax.grid(True, alpha=0.3)

In [ ]:
plt.tight_layout()
plt.savefig('../outputs/visualizations/seasonal_anomalies.png', dpi=300, bbox_inches='tight')
plt.show()

---------------------------------------------------------------------
8. ANOMALY SEVERITY RANKING
---------------------------------------------------------------------

In [ ]:
print("\n" + "="*80)
print("ANOMALY SEVERITY RANKING")
print("="*80)

In [ ]:
# Calculate anomaly severity scores
daily_metrics['severity_score'] = 0

In [ ]:
# Weighted combination of anomaly indicators
metrics_weights = {
    'orders': 0.3,
    'gmv': 0.3,
    'cancellation_rate': 0.2,
    'payment_failure_rate': 0.1,
    'active_users': 0.1
}

In [ ]:
for metric, weight in metrics_weights.items():
    if metric in daily_metrics.columns:
        # Z-score of the metric
        z_score = (daily_metrics[metric] - daily_metrics[metric].mean()) / daily_metrics[metric].std()
        daily_metrics['severity_score'] += abs(z_score) * weight

In [ ]:
# Rank anomalies by severity
daily_metrics['severity_rank'] = daily_metrics['severity_score'].rank(ascending=False)

In [ ]:
print("\n📊 Top 10 Most Severe Anomalies:")
top_anomalies = daily_metrics.nlargest(10, 'severity_score')
columns_to_show = ['date', 'orders', 'gmv', 'cancellation_rate', 'payment_failure_rate', 'severity_score']
print(top_anomalies[columns_to_show].to_string(index=False))

In [ ]:
# Visualize anomaly severity
fig, ax = plt.subplots(figsize=(14, 6))

In [ ]:
# Plot severity scores
ax.bar(daily_metrics['date'], daily_metrics['severity_score'], 
       color=daily_metrics['combined_anomaly'].map({True: '#e74c3c', False: '#3498db'}), 
       alpha=0.7)
ax.set_title('Anomaly Severity Scores Over Time', fontsize=14, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Severity Score')
ax.tick_params(axis='x', rotation=45)
ax.grid(True, alpha=0.3)

In [ ]:
# Add threshold line
threshold = daily_metrics['severity_score'].quantile(0.95)
ax.axhline(y=threshold, color='red', linestyle='--', label='95th Percentile')

In [ ]:
# Add top anomalies labels
top_5 = daily_metrics.nlargest(5, 'severity_score')
for _, row in top_5.iterrows():
    ax.text(row['date'], row['severity_score'] + 0.01, 
            f"{row['severity_rank']:.0f}", ha='center', va='bottom', fontsize=8, rotation=0)

In [ ]:
ax.legend()

In [ ]:
plt.tight_layout()
plt.savefig('../outputs/visualizations/anomaly_severity.png', dpi=300, bbox_inches='tight')
plt.show()

---------------------------------------------------------------------
9. REAL-TIME MONITORING DASHBOARD
---------------------------------------------------------------------

In [ ]:
print("\n" + "="*80)
print("REAL-TIME MONITORING DASHBOARD")
print("="*80)

In [ ]:
# Create monitoring dashboard
dashboard_data = {
    'Metric': [],
    'Current Value': [],
    'Normal Range': [],
    'Status': []
}

In [ ]:
# Get latest day
latest_day = daily_metrics.iloc[-1]
previous_day = daily_metrics.iloc[-2]

In [ ]:
metrics_monitor = {
    'Orders': 'orders',
    'GMV (₹)': 'gmv',
    'Cancellation Rate (%)': 'cancellation_rate',
    'AOV (₹)': 'aov',
    'Active Users': 'active_users',
    'Payment Failure Rate (%)': 'payment_failure_rate'
}

In [ ]:
for display_name, metric in metrics_monitor.items():
    if metric in daily_metrics.columns:
        current_value = latest_day[metric]
        mean_value = daily_metrics[metric].mean()
        std_value = daily_metrics[metric].std()
        
        # Determine if current value is anomalous
        z_score = abs((current_value - mean_value) / std_value) if std_value > 0 else 0
        
        if z_score > 3:
            status = '🔴 Critical'
        elif z_score > 2:
            status = '🟡 Warning'
        else:
            status = '🟢 Normal'
        
        dashboard_data['Metric'].append(display_name)
        dashboard_data['Current Value'].append(f"{current_value:.2f}")
        dashboard_data['Normal Range'].append(f"{mean_value - 2*std_value:.2f} - {mean_value + 2*std_value:.2f}")
        dashboard_data['Status'].append(status)

In [ ]:
dashboard_df = pd.DataFrame(dashboard_data)
print("\n📊 Real-Time Monitoring Dashboard:")
print(dashboard_df.to_string(index=False))

In [ ]:
# Weather and traffic impact
current_weather = latest_day['condition'] if 'condition' in latest_day else 'unknown'
current_traffic = latest_day['traffic_index'] if 'traffic_index' in latest_day else 0

In [ ]:
print(f"\n🌤️ Current Conditions:")
print(f"  • Weather: {current_weather}")
print(f"  • Traffic Index: {current_traffic:.2f}")

---------------------------------------------------------------------
10. BUSINESS RECOMMENDATIONS
---------------------------------------------------------------------

In [ ]:
print("\n" + "="*80)
print("BUSINESS RECOMMENDATIONS")
print("="*80)

In [ ]:
print("""
🏆 KEY INSIGHTS & RECOMMENDATIONS:
==================================

1. ANOMALY DETECTION SUMMARY:
   • Total anomalies detected: {total_anomalies}
   • Anomaly rate: {anomaly_rate:.1f}%
   • Most severe anomalies: {top_anomalies_count} days with severity score > 95th percentile
   • {structural_breaks_count} structural breaks detected in key metrics

2. ROOT CAUSE INSIGHTS:
   • {weather_anomaly_pct:.1f}% of anomalies occur during extreme weather
   • {traffic_anomaly_pct:.1f}% of anomalies correlate with high traffic
   • {weekend_anomaly_pct:.1f}% of anomalies occur on weekends
   • Most common anomaly type: {top_anomaly_type}

3. SEASONAL PATTERNS:
   • Weekly pattern explains {seasonal_explained:.1f}% of variation
   • {seasonal_anomaly_count} days deviate significantly from seasonal pattern
   • {structural_breaks_count} structural breaks indicate trend changes

🎯 ACTIONABLE RECOMMENDATIONS:
==============================

PRIORITY 1 (Immediate - Next 30 Days):
---------------------------------------
1. Implement real-time monitoring dashboard:
   • Set up alerts for critical anomalies
   • Configure notification system (email/Slack/SMS)
   • Define escalation procedures

2. Investigate root causes of recent anomalies:
   • {top_anomaly_date}: {top_anomaly_description}
   • Check for external factors (weather, events)
   • Review operational logs for issues

3. Develop anomaly response playbook:
   • Define roles and responsibilities
   • Create investigation templates
   • Establish decision-making framework

PRIORITY 2 (Short-term - Next 90 Days):
--------------------------------------
1. Enhance anomaly detection:
   • Add more data sources (app performance, support tickets)
   • Implement machine learning for anomaly classification
   • Build predictive anomaly detection

2. Automate anomaly response:
   • Automatic alerts to relevant teams
   • Self-healing mechanisms for common issues
   • Automated reporting to stakeholders

3. Analyze structural breaks:
   • Identify what caused trend changes
   • Adjust forecasts accordingly
   • Update business plans

PRIORITY 3 (Long-term - Next 6 Months):
--------------------------------------
1. Build anomaly detection API:
   • Real-time scoring for all metrics
   • Integration with monitoring systems
   • Customizable threshold settings

2. Develop predictive analytics:
   • Predict anomalies before they occur
   • Proactive risk mitigation
   • Strategic planning based on insights

📈 SUCCESS METRICS:
==================
• Anomaly detection accuracy: >95%
• Mean time to detect: <5 minutes
• Mean time to respond: <30 minutes
• False positive rate: <5%
• Business impact prevented: 20% reduction in anomaly-related losses
""".format(
    total_anomalies=daily_metrics['combined_anomaly'].sum(),
    anomaly_rate=daily_metrics['combined_anomaly'].sum() / len(daily_metrics) * 100,
    top_anomalies_count=len(daily_metrics[daily_metrics['severity_score'] > daily_metrics['severity_score'].quantile(0.95)]),
    structural_breaks_count=len([m for m in break_metrics if m in daily_metrics.columns]),
    weather_anomaly_pct=daily_metrics[daily_metrics['combined_anomaly']]['condition'].value_counts().max() / daily_metrics['combined_anomaly'].sum() * 100 if daily_metrics['combined_anomaly'].sum() > 0 else 0,
    traffic_anomaly_pct=100 if daily_metrics[daily_metrics['combined_anomaly']]['traffic_index'].mean() > daily_metrics['traffic_index'].mean() + daily_metrics['traffic_index'].std() else 0,
    weekend_anomaly_pct=daily_metrics[daily_metrics['combined_anomaly']]['is_weekend'].sum() / daily_metrics['combined_anomaly'].sum() * 100 if daily_metrics['combined_anomaly'].sum() > 0 else 0,
    top_anomaly_type=daily_metrics[daily_metrics['combined_anomaly']]['anomaly_type'].value_counts().index[0] if daily_metrics['combined_anomaly'].sum() > 0 else 'Unknown',
    seasonal_explained=1 - (daily_metrics['orders'] - seasonal_means).var() / daily_metrics['orders'].var() if len(daily_metrics) > 0 else 0,
    seasonal_anomaly_count=seasonal_anomalies.sum() if 'seasonal_anomaly' in daily_metrics.columns else 0,
    top_anomaly_date=daily_metrics.nlargest(1, 'severity_score')['date'].iloc[0].strftime('%Y-%m-%d') if len(daily_metrics) > 0 else 'N/A',
    top_anomaly_description=f"Orders: {daily_metrics.nlargest(